In [548]:
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from collections import defaultdict


## Dataset Path

In [818]:
#
adapt=20
identifier=f"30000_1000_100_adaptable{adapt}" if adapt > 0 else "30000_1000_100"
# identifier="1000_200_median"
parent_dir=os.path.join(os.getcwd(),os.pardir,os.pardir)
dataset_path=os.path.join(parent_dir,"extract_Nripples","train_pedro","dataset_up_down",identifier)
datasets=os.listdir(dataset_path)
if "config.json" in datasets:
    datasets.remove("config.json")
    

In [819]:
# prefix="updnb4ds_100_13f_60_80"
prefix=f"dsb4updn_median_200_12b"
prefix=f"{prefix}_adapt{adapt}" if adapt > 0 else prefix
SEEDED=False
if SEEDED:
    seed=0
    time_duration= 60 # in seconds, the duration of the test
    window=np.arange(seed*time_duration*1000, (time_duration+seed*time_duration)*1000, 1) # 1000 ms window
    results_json=os.path.join(os.getcwd(),f"{prefix}_results_seed{seed}.json")
    spikes=os.path.join(os.getcwd(),"spikes",f"{prefix}_spikes_seed{seed}.npy")
else:
    results_json=os.path.join(os.getcwd(),f"{prefix}_results.json")
    spikes=os.path.join(os.getcwd(),"spikes",f"{prefix}_spikes.npy")

print(f"Loading results from {results_json}")

Loading results from c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\snnTorch\eval\dsb4updn_median_200_12b_adapt20_results.json


In [820]:
# Load results
with open(results_json, 'r') as f:
    results = json.load(f)
# Load spikes
spikes = np.load(spikes, allow_pickle=True)

In [821]:
# Get some parameters
parameters=results["parameters"]
ripple_detection_offset=parameters["ripple_detection_offset"]

# refractory_period=parameters["refractory_period_gt"]
refractory_period=0 # in ms

max_detection_offset=parameters["max_detection_offset"]
print(f"Max detection offset: {max_detection_offset} ms")
# tolerance=parameters["tolerance"]
tolerance=20 # in ms
padding=100 # in ms

Max detection offset: 81 ms


In [836]:
# Read the datasets and evaluate the results
information_dataset={}
for dataset_id,dataset in enumerate(datasets):

    data_path= os.path.join(dataset_path, dataset)
    print(f"Processing dataset: {data_path}")
    data=np.load(os.path.join(data_path, "spike_data.npy"))  # Load the input data (UP/DN spikes)
    ripples=np.load(os.path.join(data_path, "ripples.npy"))  # Load the GT data (HFO Insertion Timing)
    dataset_results = results[dataset]  # Get the results for the current dataset

    # ripples_start = ripples[:, 0]  # Get the GT Insertion Timing (first column of the ripples array)
    if SEEDED:
        # If the data is seeded, we need to slice the data to the desired time window
        data = data[window,:]
        ripples_window = []
        for ripple in ripples:
            if ripple[1] >= window[0] and ripple[0] <= window[-1]:
                    ripples_window.append(ripple)
        ripples=np.array(ripples_window)-window[0]-tolerance  # Adjust the GT Insertion Timing to the new window
    else:
        ripples=ripples-tolerance
    
    num_hfo_events = len(ripples)
    dataset_spikes= spikes[dataset_id*8:(dataset_id+1)*8,:]

    ripple_hits = {}  # List of sets: which channels hit each ripple
    undetected_ripples = []  # Indices of ripples missed by all channels
    num_channels = dataset_spikes.shape[0]
    filtered_spikes = []  # List of filtered spikes for each channel
    
    # GET VALID SPIKES by REMOVING REFRACTORY PERIOD VIOLATIONS
    for ch_spikes in dataset_spikes:
        # Sort spikes just in case (important)
        sorted_spikes = np.sort(ch_spikes)
        
        if len(sorted_spikes) == 0:
            filtered_spikes.append(np.array([]))
            continue

        valid = [sorted_spikes[0]]
        for spike in sorted_spikes[1:]:
            if spike - valid[-1] >= refractory_period:
                valid.append(spike)
        filtered_spikes.append(np.array(valid))

    ripples_start = ripples[:, 0]  # Get the GT Insertion Timing (first column of the ripples array)
    # DETECT RIPPLE HITS
    for idx, ripple_time in enumerate(ripples_start):
        detected_channels = set()

        for ch in range(num_channels):
            valid_window_start= ripple_time - tolerance
            valid_window_end = ripple_time + max_detection_offset + tolerance
            ch_spikes = filtered_spikes[ch]
            in_window = np.any((ch_spikes >= valid_window_start) & 
                            (ch_spikes <= valid_window_end))
            if in_window:
                detected_channels.add(ch)

        ripple_hits[idx] = detected_channels  # store in dict by ripple index

        if not detected_channels:
            undetected_ripples.append(idx)

    # COLLECT FALSE POSITIVE SPIKES
    valid_detection_ranges = []

    for ripple_time in ripples:
        start = ripple_time[0] - tolerance
        end = ripple_time[0] + max_detection_offset + tolerance + padding
        # end=max(end,ripple_time[1] + padding)  # Ensure end is at least ripple_time[1] + padding
        valid_detection_ranges.append((start, end))

    # STEP 2: Function to check if a spike is inside any ripple detection window
    def is_tp(spike_time, valid_ranges):
        for start, end in valid_ranges:
            if start <= spike_time <= end:
                return True
        return False
    
    fp_spikes_by_channel = defaultdict(list)
    for ch in range(num_channels):
        # STEP 1: Create set of all valid detection time ranges
        for spike_time in filtered_spikes[ch]:
            if not is_tp(spike_time, valid_detection_ranges):
                fp_spikes_by_channel[ch].append(spike_time)
    
    # Group by time with jitter
    overlap_counts = defaultdict(set)  # spike_time -> set of channels that had an FP there
    fp_timing_to_channels = defaultdict(set)
    jitter= np.round(max_detection_offset) # half of the max detection offset
    for ch, spike_list in fp_spikes_by_channel.items():
        for spike_time in spike_list:
            matched_time = None

            # Try to match this spike to an existing FP group (within jitter)
            for t in range(int(spike_time - jitter),int( spike_time + jitter + 1)):
                if t in fp_timing_to_channels:
                    matched_time = t
                    break
            if matched_time is not None:
                fp_timing_to_channels[matched_time].add(ch)
            else:
                fp_timing_to_channels[spike_time].add(ch)

    information_dataset[dataset] = {
    "ripple_hits": ripple_hits,  # now a dict: ripple_idx -> set of channels
    "undetected_ripples": undetected_ripples,  # list of ints
    "filtered_spikes": filtered_spikes,  # list of arrays
    "fp_spikes_by_channel": fp_spikes_by_channel,  # dict: ch -> [spike times]
    "fp_timing_to_channels": dict(fp_timing_to_channels),  # dict: time -> set(channels)
    }


Processing dataset: c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\snnTorch\eval\..\..\extract_Nripples\train_pedro\dataset_up_down\30000_1000_100_adaptable20\Amigo2_2019-07-11_11-57-07
Processing dataset: c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\snnTorch\eval\..\..\extract_Nripples\train_pedro\dataset_up_down\30000_1000_100_adaptable20\Dlx1_2021-02-12_12-46-54
Processing dataset: c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\snnTorch\eval\..\..\extract_Nripples\train_pedro\dataset_up_down\30000_1000_100_adaptable20\Som2_2019-07-24_12-01-49
Processing dataset: c:\Users\NCN\Documents\PedroFelix\LAVA_SNN_ripples\snnTorch\eval\..\..\extract_Nripples\train_pedro\dataset_up_down\30000_1000_100_adaptable20\Thy7_2020-11-11_16-05-00


In [837]:
# TP for each dataset:
min_detected_channels = 5 # Change this value as needed

tp_per_dataset = []
fn_per_dataset = []
fp_per_dataset = []

###
tp_all_dataset=[]
fp_all_dataset=[]
fn_all_dataset=[]

for dataset, info in information_dataset.items():
    ripple_hits = info["ripple_hits"]  # dict: ripple_idx -> set of channels detecting each ripple
    fp_timing_to_channels = info.get("fp_timing_to_channels", {})  # dict: spike_time -> set of channels
    
    # True Positives: ripples detected by at least min_detected_channels channels
    tp_count = sum(1 for hits in ripple_hits.values() if len(hits) >= min_detected_channels)
    tp_all=sum(len(hits) for hits in ripple_hits.values())

    tp_per_dataset.append(tp_count)
    tp_all_dataset.append(tp_all)


    # False Negatives: ripples detected by fewer than min_detected_channels channels (including zero)
    fn_count = sum(1 for hits in ripple_hits.values() if len(hits) < min_detected_channels)
    fn_all=sum(8- len(hits) for hits in ripple_hits.values())
    # print(fn_all)
    
    fn_per_dataset.append(fn_count)
    fn_all_dataset.append(fn_all)

    # Overlap counts: how many FP timings have at least one channel
    fp_counts = sum(1 for channels in fp_timing_to_channels.values() if len(channels) >=min_detected_channels )
    # print(fp_counts)
    fp_all=sum(len(channels) for channels in fp_timing_to_channels.values() )
    
    fp_per_dataset.append(fp_counts)
    fp_all_dataset.append(fp_all)

In [838]:
for dataset, info in information_dataset.items():
    tp_dataset=0
    fp_dataset=0
    fn_dataset=0
    ripple_hits = info["ripple_hits"]  # dict: ripple_idx -> set of channels detecting each ripple
    fp_timing_to_channels = info.get("fp_timing_to_channels", {})  # dict: spike_time -> set of channels
    print(f"Dataset: {dataset}")
    for channel in range(8):
        # True Positives: ripples detected by at least min_detected_channels channels
        tp_channel = sum(1 for hits in ripple_hits.values() if channel in hits)


        # False Negatives: ripples detected by fewer than min_detected_channels channels (including zero)
        fn_channel = sum(1 for hits in ripple_hits.values() if channel not in hits)
    
        # FP channels:
        fp_channel = sum(1 for channels in fp_timing_to_channels.values() if channel in channels)
        # Overlap counts: how many FP timings have at least one channel
        precision= tp_channel / (tp_channel + fp_channel) if (tp_channel + fp_channel) > 0 else 0
        recall= tp_channel / (tp_channel + fn_channel) if (tp_channel + fn_channel) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        tp_dataset+=tp_channel
        fp_dataset+=fp_channel
        fn_dataset+=fn_channel
        print(f"Channel {channel}: TP={tp_channel}, FN={fn_channel}, FP={fp_channel} || ",f"Precision={precision:.2f}, Recall={recall:.2f}, F1-Score={f1_score:.2f}")
    precision_dataset = tp_dataset / (tp_dataset + fp_dataset) if (tp_dataset + fp_dataset) > 0 else 0
    recall_dataset = tp_dataset / (tp_dataset + fn_dataset) if (tp_dataset + fn_dataset) > 0 else 0
    f1_score_dataset = 2 * (precision_dataset * recall_dataset) / (precision_dataset + recall_dataset) if (precision_dataset + recall_dataset) > 0 else 0
    print(f"Summary: TP={tp_dataset}, FN={fn_dataset}, FP={fp_dataset} || Precision={precision_dataset:.2f}, Recall={recall_dataset:.2f}, F1-Score={f1_score_dataset:.2f}\n")

Dataset: Amigo2_2019-07-11_11-57-07
Channel 0: TP=141, FN=83, FP=107 ||  Precision=0.57, Recall=0.63, F1-Score=0.60
Channel 1: TP=138, FN=86, FP=105 ||  Precision=0.57, Recall=0.62, F1-Score=0.59
Channel 2: TP=113, FN=111, FP=62 ||  Precision=0.65, Recall=0.50, F1-Score=0.57
Channel 3: TP=104, FN=120, FP=48 ||  Precision=0.68, Recall=0.46, F1-Score=0.55
Channel 4: TP=65, FN=159, FP=38 ||  Precision=0.63, Recall=0.29, F1-Score=0.40
Channel 5: TP=59, FN=165, FP=22 ||  Precision=0.73, Recall=0.26, F1-Score=0.39
Channel 6: TP=33, FN=191, FP=24 ||  Precision=0.58, Recall=0.15, F1-Score=0.23
Channel 7: TP=20, FN=204, FP=30 ||  Precision=0.40, Recall=0.09, F1-Score=0.15
Summary: TP=673, FN=1119, FP=436 || Precision=0.61, Recall=0.38, F1-Score=0.46

Dataset: Dlx1_2021-02-12_12-46-54
Channel 0: TP=16, FN=15, FP=6 ||  Precision=0.73, Recall=0.52, F1-Score=0.60
Channel 1: TP=22, FN=9, FP=12 ||  Precision=0.65, Recall=0.71, F1-Score=0.68
Channel 2: TP=23, FN=8, FP=8 ||  Precision=0.74, Recall=0.74

In [839]:
from collections import Counter
def count_by_num_channels(sets_list_or_dict):
    """
    Input:
        - sets_list_or_dict: list or dict of sets (e.g., ripple_hits or fp_timing_to_channels)
    Output:
        - Counter mapping: number of channels → count of events with that many channels
    """
    if isinstance(sets_list_or_dict, dict):
        sets_iter = sets_list_or_dict.values()
    else:
        sets_iter = sets_list_or_dict

    counts = Counter()
    for ch_set in sets_iter:
        n = len(ch_set)
        counts[n] += 1
    return counts

# Add sharing stats to each dataset
for dataset, info in information_dataset.items():
    ripple_hits = info["ripple_hits"]  # dict: ripple_index → set(channels)
    fp_timing_to_channels = info.get("fp_timing_to_channels", {})  # dict: time → set(channels)

    # Compute counts
    ripple_sharing_counts = count_by_num_channels(ripple_hits)
    fp_sharing_counts = count_by_num_channels(fp_timing_to_channels)

    # Store in original dataset info
    info["ripple_sharing_counts"] = ripple_sharing_counts
    info["fp_sharing_counts"] = fp_sharing_counts

    # Optional: print summary
    print(f"Dataset: {dataset}")
    print("Ripples shared across channels:")
    for channels_num in range(1, num_channels + 1):
        print(f"  Shared by {channels_num} channel(s): {ripple_sharing_counts.get(channels_num, 0)}")

    print("False positives shared across channels:")
    for channels_num in range(1, num_channels + 1):
        print(f"  Shared by {channels_num} channel(s): {fp_sharing_counts.get(channels_num, 0)}")


Dataset: Amigo2_2019-07-11_11-57-07
Ripples shared across channels:
  Shared by 1 channel(s): 37
  Shared by 2 channel(s): 28
  Shared by 3 channel(s): 25
  Shared by 4 channel(s): 35
  Shared by 5 channel(s): 18
  Shared by 6 channel(s): 19
  Shared by 7 channel(s): 15
  Shared by 8 channel(s): 7
False positives shared across channels:
  Shared by 1 channel(s): 155
  Shared by 2 channel(s): 41
  Shared by 3 channel(s): 27
  Shared by 4 channel(s): 15
  Shared by 5 channel(s): 5
  Shared by 6 channel(s): 2
  Shared by 7 channel(s): 3
  Shared by 8 channel(s): 0
Dataset: Dlx1_2021-02-12_12-46-54
Ripples shared across channels:
  Shared by 1 channel(s): 0
  Shared by 2 channel(s): 1
  Shared by 3 channel(s): 2
  Shared by 4 channel(s): 3
  Shared by 5 channel(s): 4
  Shared by 6 channel(s): 4
  Shared by 7 channel(s): 7
  Shared by 8 channel(s): 4
False positives shared across channels:
  Shared by 1 channel(s): 19
  Shared by 2 channel(s): 7
  Shared by 3 channel(s): 2
  Shared by 4 cha

In [840]:
#### Calculate metrics per dataset based on TP, FN, FP counts
metrics_per_dataset = []
keys= information_dataset.keys()
keys_minus_overall = [k for k in keys if k != "overall_metrics"]
for i, dataset_name in enumerate(keys_minus_overall):
    tp = tp_per_dataset[i]
    fn = fn_per_dataset[i]
    fp = fp_per_dataset[i]  
    # Calculate metrics
    tp_all = tp_all_dataset[i]
    fn_all = fn_all_dataset[i]
    fp_all = fp_all_dataset[i]

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    fdr = fp / (tp + fp) if (tp + fp) > 0 else 0.0

    precision_all = tp_all / (tp_all + fp_all) if (tp_all + fp_all) > 0 else 0.0
    recall_all = tp_all / (tp_all + fn_all) if (tp_all + fn_all) > 0 else 0.0
    f1_score_all = (2 * precision_all * recall_all) / (precision_all + recall_all) if (precision_all + recall_all) > 0 else 0.0
    metrics = {
        "dataset": dataset_name,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "Precision": precision,
        "Recall": recall,
        "F1": f1_score,
        "FDR": fdr,
        "TP_all": tp_all,
        "FN_all": fn_all,
        "FP_all": fp_all,
        "precision_all": precision_all,
        "recall_all": recall_all,
        "f1_score_all": f1_score_all,
    }

    metrics_per_dataset.append(metrics)
    information_dataset[dataset_name]["metrics"] = metrics  # save in original dict

# Optional: print summary
for m in metrics_per_dataset:
    print(f"{m['dataset']}: Precision={m['Precision']:.4f}, Recall={m['Recall']:.4f}, F1={m['F1']:.4f}, FDR={m['FDR']:.4f}")
    print(f"  TP={m['TP']}, FN={m['FN']}, FP={m['FP']}")
    



Amigo2_2019-07-11_11-57-07: Precision=0.8551, Recall=0.2634, F1=0.4027, FDR=0.1449
  TP=59, FN=165, FP=10
Dlx1_2021-02-12_12-46-54: Precision=0.8636, Recall=0.6129, F1=0.7170, FDR=0.1364
  TP=19, FN=12, FP=3
Som2_2019-07-24_12-01-49: Precision=0.9394, Recall=0.4844, F1=0.6392, FDR=0.0606
  TP=31, FN=33, FP=2
Thy7_2020-11-11_16-05-00: Precision=0.9310, Recall=0.1942, F1=0.3214, FDR=0.0690
  TP=27, FN=112, FP=2


In [841]:
for m in metrics_per_dataset:
    print(f"{m['dataset']}: TP_all={m['TP_all']}, FN_all={m['FN_all']}, FP_all={m['FP_all']}")
    print(f"Precision_all={m['precision_all']:.4f}, Recall_all={m['recall_all']:.4f}, F1_all={m['f1_score_all']:.4f}")

Amigo2_2019-07-11_11-57-07: TP_all=673, FN_all=1119, FP_all=436
Precision_all=0.6069, Recall_all=0.3756, F1_all=0.4640
Dlx1_2021-02-12_12-46-54: TP_all=145, FN_all=103, FP_all=59
Precision_all=0.7108, Recall_all=0.5847, F1_all=0.6416
Som2_2019-07-24_12-01-49: TP_all=262, FN_all=250, FP_all=44
Precision_all=0.8562, Recall_all=0.5117, F1_all=0.6406
Thy7_2020-11-11_16-05-00: TP_all=352, FN_all=760, FP_all=36
Precision_all=0.9072, Recall_all=0.3165, F1_all=0.4693


In [842]:
# Total counts
total_tp = sum(tp_per_dataset)
total_fn = sum(fn_per_dataset)
total_fp = sum(fp_per_dataset)
total_tp_all = sum(tp_all_dataset)
total_fn_all = sum(fn_all_dataset)
total_fp_all = sum(fp_all_dataset)

# Overall metrics
overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
overall_f1 = (2 * overall_precision * overall_recall) / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0.0
overall_fdr = total_fp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0

# Overall metrics all
overall_precision_all = total_tp_all / (total_tp_all + total_fp_all) if (total_tp_all + total_fp_all) > 0 else 0.0
overall_recall_all = total_tp_all / (total_tp_all + total_fn_all) if (total_tp_all + total_fn_all) > 0 else 0.0
overall_f1_all = (2 * overall_precision_all * overall_recall_all) / (overall_precision_all + overall_recall_all) if (overall_precision_all + overall_recall_all) > 0 else 0.0

# Store in dict
overall_metrics = {
    "Total TP": total_tp,
    "Total FN": total_fn,
    "Total FP": total_fp,
    "Precision": overall_precision,
    "Recall": overall_recall,
    "F1 Score": overall_f1,
    "FDR": overall_fdr,
}
information_dataset["overall_metrics"] = overall_metrics
# Print nicely
print("\n--- Overall Metrics Across All Datasets ---")
for k, v in overall_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

print("\n--- Overall Metrics Across All Datasets (All) ---")
overall_metrics_all = {
    "Total TP_all": total_tp_all,
    "Total FN_all": total_fn_all,
    "Total FP_all": total_fp_all,
    "Precision_all": overall_precision_all,
    "Recall_all": overall_recall_all,
    "F1 Score_all": overall_f1_all,
}
for k, v in overall_metrics_all.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
    


--- Overall Metrics Across All Datasets ---
Total TP: 136
Total FN: 322
Total FP: 17
Precision: 0.8889
Recall: 0.2969
F1 Score: 0.4452
FDR: 0.1111

--- Overall Metrics Across All Datasets (All) ---
Total TP_all: 1432
Total FN_all: 2232
Total FP_all: 575
Precision_all: 0.7135
Recall_all: 0.3908
F1 Score_all: 0.5050


In [770]:
# import pickle
# path=os.path.join(os.getcwd(), "information_dataset_channels")
# os.makedirs(path, exist_ok=True)
# file_name=f"{prefix}_information_dataset.pkl" if not SEEDED else f"{prefix}_information_dataset_seed{seed}.pkl"
# with open(os.path.join(path,file_name), "wb") as f:
#     pickle.dump(information_dataset, f)